# Verify the numbers of the paper

Recomputes the paper's resolved counts, paired bootstrap intervals, termination rates and labels, and the external-panel repair counts from the per-task tables of the package, and compares them with the sealed reports. Python 3.10+ and numpy only, no internet access needed when the package is attached as a Kaggle dataset or sits next to this notebook. Nothing is downloaded or written.

In [ ]:
import glob, os, sys

def find_package():
    candidates = [os.environ.get("PAPER_PACKAGE", ""), os.getcwd(), os.path.dirname(os.getcwd())]
    candidates += glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")
    for c in candidates:
        if c and os.path.isfile(os.path.join(c, "verify", "verify_paper_numbers.py")):
            return c
    raise FileNotFoundError("package not found: attach it as a dataset, run from its folder, or set PAPER_PACKAGE")

ROOT = find_package()
sys.path.insert(0, os.path.join(ROOT, "verify"))
print("package:", ROOT)

## 1. Test the functions on a hand-made table (no project data)

In [ ]:
import synthetic_fixture
assert synthetic_fixture.main() == 0, "fixture failed"

## 2. Recompute the paper's numbers from the package

In [ ]:
import verify_paper_numbers as vp
res = vp.run_checks(ROOT)
width = max(len(r["name"]) for r in res.rows)
for r in res.rows:
    detail = r["note"] if r["status"] == "NOT RECOMPUTABLE" else "recomputed=%s | sealed=%s%s" % (r["recomputed"], r["sealed"], " | paper=" + r["paper"] if r["paper"] else "")
    print("%-16s %-*s  %s" % (r["status"], width, r["name"], detail[:160]))
c = res.counts()
print()
print("PASS %d   FAIL %d   NOT RECOMPUTABLE %d" % (c["PASS"], c["FAIL"], c["NOT RECOMPUTABLE"]))
assert c["FAIL"] == 0

A line marked NOT RECOMPUTABLE is not a pass: the number needs data that are not in the package (agent traces, or analyses that were never run with a per-task trace table). `FAIL` would mean a recomputed value differs from the sealed report or from the number printed in the paper.